# Total Snippets Extraction
This notebook extract all opium mentions from the full library to prepare opium snippets for topic modelling (see 3_analysis).

In [ ]:
import os
import re
import pandas as pd
import spacy
import ipywidgets as widgets
from tqdm.auto import tqdm

from IPython.display import display, HTML

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')
    
import sys
sys.path.append("..")

from utils.constants import KEYWORDS

In [10]:
# Setup Paths
metadata_path = '../../data/metadata_files/GP_opium_filtered_1870_1920.parquet' # Changed this
fulltext_path = '../../../../../Downloads/history/opium_books_fulltext'

output_parquet = '../../data/snippets/total_snippets.parquet'
output_csv = '../../data/snippets/total_snippets.csv'

window_size = 100

## 1. Extracting context windows out of all books

In [11]:
book_ids = pd.read_parquet(metadata_path)["Etext Number"]
book_ids

0          16
1          24
2          27
3          36
4          44
        ...  
3738    74736
3739    74886
3740    74956
3741    75246
3742    75497
Name: Etext Number, Length: 3743, dtype: int64

In [ ]:
results = []
if os.path.exists(fulltext_path):
    for book_id in book_ids:
    #os.listdir(metadata_path):
        filepath = os.path.join(fulltext_path, str(book_id))
        if os.path.isfile(filepath):
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                text_lower = text.lower()
                for keyword in KEYWORDS:
                    pattern = r'\b' + re.escape(keyword) + r'\b'
                    for match in re.finditer(pattern, text_lower):
                        idx = match.start()
                        left_text = text[:idx]
                        right_text = text[idx + len(keyword):]
                        
                        left_words = left_text.split()
                        right_words = right_text.split()
                        
                        context_left = ' '.join(left_words[-window_size:])
                        context_right = ' '.join(right_words[:window_size])
                        
                        # Simple bounding box handling for text window
                        text_start = max(0, idx - 800)
                        text_end = min(len(text), idx + len(keyword) + 800)
                        raw_snippet = text[text_start:text_end] 
                        
                        results.append({
                            'Book_ID': book_id,
                            'Keyword': keyword,
                            'Left_Context': context_left,
                            'Right_Context': context_right,
                            'Full_Context': f"{context_left} {text[idx:idx+len(keyword)]} {context_right}"
                        })
                        
df = pd.DataFrame(results)
print(f"Found {len(df)} keyword mentions across all {len(book_ids)} books.")


Found 3857 keyword mentions across all 3743 books.


In [20]:
# To overwrite the total snippet files, uncomment:

df[['Book_ID', 'Keyword', 'Full_Context']].to_csv(output_csv)
df[['Book_ID', 'Keyword', 'Full_Context']].to_parquet(output_parquet)

## 2. Manual Inspection Viewer
Use the slider to browse through the extracted snippets. The target keyword is highlighted.

In [16]:
pd.set_option('display.max_colwidth', None)
def view_snippet(index):
    if len(df) == 0:
        print("No snippets found.")
        return
    row = df.iloc[index]
    html_out = f"""
    <div style='font-family: Georgia, serif; font-size: 16px; line-height: 1.6; max-width: 800px; padding: 20px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;'>
        <h4>Book ID: {row['Book_ID']} | Keyword: <span style='color: dimgrey;'>{row['Keyword'].upper()}</span></h4>
        <hr>
        <p>
            {row['Left_Context']} 
            <span style='background-color: #ffeb3b; font-weight: bold; padding: 0 4px;'>{row['Keyword']}</span> 
            {row['Right_Context']}
        </p>
    </div>
    """
    display(HTML(html_out))

In [ ]:
if len(df) > 0:
    slider = widgets.IntSlider(min=0, max=len(df)-1, step=1, description='Snippet:', layout=widgets.Layout(width='800px'))
    widgets.interact(view_snippet, index=slider)

interactive(children=(IntSlider(value=0, description='Snippet:', layout=Layout(width='800px'), max=3856), Outp…

## 3. Entity & POS Exploration (Optional)
Analyze the named entities (PEOPLE, ORG, LOC) and adjectives co-occurring with opium keywords across the entire dataset.

In [ ]:
all_entities = []
all_adjectives = []

for text in tqdm(df['Full_Context'], desc="Processing NLP Contexts"):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ['PERSON', 'ORG', 'GPE', 'LOC', 'FAC', 'PRODUCT']:
            all_entities.append((ent.label_, ent.text.strip()))
    for token in doc:
        if token.pos_ == 'ADJ' and not token.is_stop and token.is_alpha:
            all_adjectives.append(token.lemma_.lower())

ent_counts = Counter(all_entities).most_common(20)
adj_counts = Counter(all_adjectives).most_common(20)

print("\n--- Top 20 Named Entities in Contexts ---")
for e, c in ent_counts:
    print(f"  {e[0]:<7}: {e[1]} ({c})")

print("\n--- Top 20 Adjectives in Contexts ---")
for a, c in adj_counts:
    print(f"  {a:<15}: {c}")


Processing NLP Contexts:   0%|          | 0/3857 [00:00<?, ?it/s]


--- Top 20 Named Entities in Contexts ---
  PERSON : Poppy (555)
  PERSON : Iglesias (157)
  GPE    : London (142)
  PERSON : Rickman (105)
  GPE    : China (92)
  PERSON : Rita (85)
  PERSON : Kennedy (83)
  PERSON : Bliss (83)
  PERSON : Chinaman (79)
  PERSON : Medjora (73)
  GPE    : New York (71)
  PERSON : Munson (70)
  PERSON : Sin (69)
  GPE    : England (68)
  PERSON : De Quincey (64)
  PERSON : Dominic Iglesias (63)
  PERSON : Grace (61)
  GPE    : Paris (59)
  PERSON : Ricky (59)
  PERSON : Jim (58)

--- Top 20 Adjectives in Contexts ---
  little         : 1376
  good           : 888
  old            : 874
  great          : 761
  black          : 596
  long           : 475
  white          : 469
  young          : 428
  new            : 368
  small          : 364
  bad            : 332
  poor           : 299
  poppy          : 288
  red            : 274
  dark           : 261
  certain        : 248
  dear           : 246
  dead           : 237
  open           : 235
  heav